# 7.5 — The Cost of a Grid Search

**Chapter 7, section 7.3.3.** The starting point for **Exercise 2**.

**The question this notebook answers:** which of the three responses to an expensive grid search
actually reduces the work?

The chapter names three: cache the search input, raise `parallelism`, and shrink the grid. It
argues that only the last one changes how much work there is, and that the first two change how
long that work takes. This notebook runs the same search four ways and measures both — the
wall-clock time and the number of Spark jobs the search submits.

Two things make this notebook heavier than the others in the chapter, and both are deliberate.
It fits fifty-two gradient-boosted trees, because a claim about cost cannot be shown on ten rows;
and it quotes timings, which are machine-dependent. **The ratios are the result, not the
seconds.** The job counts are not machine-dependent, and they are the sharper of the two
instruments.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile
from pyspark.sql import SparkSession

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-7.5")
         .master("local[4]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.ui.retainedJobs", "10000")   # a search submits hundreds; keep them all
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Spark", spark.version, "| cores:", sc.defaultParallelism)

Spark 4.2.0 | cores: 4


---

## 1. The instrument, and what it can and cannot see

Two numbers are recorded per search.

* **Wall-clock seconds.** Machine-dependent, and sensitive to everything else running. Useful
  only as a ratio between two runs on the same machine, minutes apart.
* **Spark jobs submitted.** Read from the driver's own status store, which is the same data the
  Spark UI's job list shows. This counts *jobs*, not tasks and not bytes: it is a measure of how
  much work was requested, and it is unaffected by how fast the machine is or by how many of
  those jobs ran at the same time.

That second property is what makes the job count the right instrument for this notebook. If
`parallelism` reduced the total work, the job count would fall. If it only reschedules the same
work, the job count stays put and the clock moves.

In [2]:
status_store = sc._jsc.sc().statusStore()


def jobs_submitted():
    """Total Spark jobs this application has submitted so far."""
    return status_store.jobsList(None).size()


print("jobs so far:", jobs_submitted())

jobs so far: 0


---

## 2. The pipeline and the grid

The bike-sharing pipeline of notebook 7.1, with a cross-validator added as the last stage. The
nesting is the chapter's: the **cross-validator goes inside the pipeline**, not the pipeline
inside the cross-validator, so the assembler and the indexer run once and the search covers only
the regressor.

In [3]:
import pyspark.sql.functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, VectorIndexer
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

df = (spark.read.csv(os.path.join(DATA, "bike-sharing.csv"), header=True, inferSchema=True)
      .drop("instant", "dteday", "casual", "registered"))
train, test = df.randomSplit([0.7, 0.3], seed=0)
print("training rows:", train.count())

featuresCols = [c for c in df.columns if c != "cnt"]
assembler = VectorAssembler(inputCols=featuresCols, outputCol="rawFeatures")
indexer = VectorIndexer(inputCol="rawFeatures", outputCol="features", maxCategories=4)
gbt = GBTRegressor(featuresCol="features", labelCol="cnt", seed=0)

evaluator = RegressionEvaluator(metricName="rmse",
                                labelCol=gbt.getLabelCol(),
                                predictionCol=gbt.getPredictionCol())

training rows: 12081


In [4]:
paramGrid = (ParamGridBuilder()
             .addGrid(gbt.maxDepth, [2, 5])
             .addGrid(gbt.maxIter, [5, 10])
             .build())

FOLDS = 3
print(f"parameter maps:        {len(paramGrid)}")
print(f"folds:                 {FOLDS}")
print(f"model fits per search: {len(paramGrid)} x {FOLDS} = {len(paramGrid) * FOLDS}"
      f", plus 1 refit on the whole training set = {len(paramGrid) * FOLDS + 1}")

parameter maps:        4
folds:                 3
model fits per search: 4 x 3 = 12, plus 1 refit on the whole training set = 13


The chapter's own grid is `maxDepth [2, 5]` by `maxIter [10, 100]`. This notebook keeps the two
depths and uses `maxIter [5, 10]` instead, so that four searches finish inside the five-minute
budget. **The arithmetic is unchanged** — four parameter maps either way — and the arithmetic is
what section 7.3.3 is about. Only the cost of an individual fit is smaller.

---

## 3. Four searches over one grid

The same estimator, the same grid, the same folds, the same seed. What varies is whether the
training DataFrame is cached and what `parallelism` is set to.

In [5]:
import time


def search(cached, parallelism):
    """Run the whole pipeline-with-cross-validator once and report cost."""
    train.unpersist(blocking=True)
    if cached:
        train.cache().count()          # materialize before the clock starts

    cv = CrossValidator(estimator=gbt, evaluator=evaluator,
                        estimatorParamMaps=paramGrid,
                        numFolds=FOLDS, parallelism=parallelism, seed=1)
    pipeline = Pipeline(stages=[assembler, indexer, cv])

    jobs_before = jobs_submitted()
    started = time.time()
    model = pipeline.fit(train)
    seconds = time.time() - started
    jobs = jobs_submitted() - jobs_before

    best = model.stages[-1].bestModel
    return {"cached": cached, "parallelism": parallelism,
            "seconds": seconds, "jobs": jobs,
            "rmse": evaluator.evaluate(model.transform(test)),
            "maxDepth": best.getMaxDepth(), "maxIter": best.getMaxIter()}


runs = [search(cached, p) for cached in (False, True) for p in (1, 4)]
for r in runs:
    print(r)

{'cached': False, 'parallelism': 1, 'seconds': 9.719703912734985, 'jobs': 417, 'rmse': 73.05381110762868, 'maxDepth': 5, 'maxIter': 10}
{'cached': False, 'parallelism': 4, 'seconds': 4.047818899154663, 'jobs': 417, 'rmse': 73.05381110762868, 'maxDepth': 5, 'maxIter': 10}
{'cached': True, 'parallelism': 1, 'seconds': 7.214982032775879, 'jobs': 417, 'rmse': 73.05381110762868, 'maxDepth': 5, 'maxIter': 10}
{'cached': True, 'parallelism': 4, 'seconds': 3.6024129390716553, 'jobs': 417, 'rmse': 73.05381110762868, 'maxDepth': 5, 'maxIter': 10}


In [6]:
header = f"{'cached':>7}{'parallelism':>13}{'seconds':>10}{'Spark jobs':>13}{'best RMSE':>12}"
print(header)
print("-" * len(header))
for r in runs:
    print(f"{str(r['cached']):>7}{r['parallelism']:>13}{r['seconds']:>10.1f}"
          f"{r['jobs']:>13}{r['rmse']:>12.2f}")

 cached  parallelism   seconds   Spark jobs   best RMSE
-------------------------------------------------------
  False            1       9.7          417       73.05
  False            4       4.0          417       73.05
   True            1       7.2          417       73.05
   True            4       3.6          417       73.05


---

## 4. Reading the table

Three readings, in the order the chapter puts them.

In [7]:
uncached_serial = next(r for r in runs if not r["cached"] and r["parallelism"] == 1)
cached_serial = next(r for r in runs if r["cached"] and r["parallelism"] == 1)
cached_parallel = next(r for r in runs if r["cached"] and r["parallelism"] == 4)

print(f"caching, at parallelism 1:  "
      f"{uncached_serial['seconds']:.1f}s -> {cached_serial['seconds']:.1f}s  "
      f"({100 * (1 - cached_serial['seconds'] / uncached_serial['seconds']):.0f}% faster), "
      f"jobs {uncached_serial['jobs']} -> {cached_serial['jobs']}")

print(f"parallelism 1 -> 4, cached: "
      f"{cached_serial['seconds']:.1f}s -> {cached_parallel['seconds']:.1f}s  "
      f"({cached_serial['seconds'] / cached_parallel['seconds']:.1f}x faster), "
      f"jobs {cached_serial['jobs']} -> {cached_parallel['jobs']}")

caching, at parallelism 1:  9.7s -> 7.2s  (26% faster), jobs 417 -> 417
parallelism 1 -> 4, cached: 7.2s -> 3.6s  (2.0x faster), jobs 417 -> 417


In [8]:
assert len({r["jobs"] for r in runs}) == 1, "the four searches did not submit the same work"
print(f"all four searches submitted exactly {runs[0]['jobs']} Spark jobs")

all four searches submitted exactly 417 Spark jobs


In [9]:
assert len({round(r["rmse"], 6) for r in runs}) == 1
print(f"all four searches selected the same model: "
      f"maxDepth={runs[0]['maxDepth']}, maxIter={runs[0]['maxIter']}, "
      f"test RMSE {runs[0]['rmse']:.2f}")

all four searches selected the same model: maxDepth=5, maxIter=10, test RMSE 73.05


**Caching moves the clock and not the job count.** The same jobs are submitted either way; the
uncached run makes each of them do more, because a Spark DataFrame is a lineage rather than
materialized data and every fold of every parameter map rebuilds it from the CSV. Nothing about
the search changed — only how much of the work each job repeats. This is why the chapter calls
caching a prerequisite rather than an optimization: it removes waste, it does not remove work.

**`parallelism` moves the clock and not the job count either.** Four models train at once instead
of one after another. The total is identical, which is exactly what `fig:GridSearchCost` claims,
and the speed-up is less than fourfold because the four concurrent fits share the same executors
and the same memory pool. On a machine with less headroom the same setting can make a search
slower, by forcing spill.

**All four searches choose the same model.** They must: the folds, the seed and the grid are the
same, so the arithmetic is the same. Caching and parallelism are cost decisions with no effect on
the answer, which is the reason they are safe to reach for first.

---

## 5. The third response, which does change the work

The grid is the only one of the three that changes the number of model fits. Dropping
`maxIter` from the grid and pinning it at 10 leaves two parameter maps instead of four — the
same two maps that were already in the larger grid, so this is a strict subset of the work just
measured.

In [10]:
# baseOn pins maxIter for every map, so the two remaining maps are exactly the maxIter=10
# half of the grid above. Without it the parameter would fall back to its default of 20 and
# the smaller grid would run *more* work per fit than the larger one.
small_grid = (ParamGridBuilder()
              .baseOn({gbt.maxIter: 10})
              .addGrid(gbt.maxDepth, [2, 5])
              .build())

train.unpersist(blocking=True)
train.cache().count()

cv_small = CrossValidator(estimator=gbt, evaluator=evaluator,
                          estimatorParamMaps=small_grid,
                          numFolds=FOLDS, parallelism=1, seed=1)
jobs_before, started = jobs_submitted(), time.time()
Pipeline(stages=[assembler, indexer, cv_small]).fit(train)
small = {"seconds": time.time() - started, "jobs": jobs_submitted() - jobs_before}

print(f"{'grid':<22}{'maps':>6}{'fits':>7}{'seconds':>10}{'jobs':>8}")
print(f"{'maxDepth x maxIter':<22}{len(paramGrid):>6}{len(paramGrid) * FOLDS + 1:>7}"
      f"{cached_serial['seconds']:>10.1f}{cached_serial['jobs']:>8}")
print(f"{'maxDepth only':<22}{len(small_grid):>6}{len(small_grid) * FOLDS + 1:>7}"
      f"{small['seconds']:>10.1f}{small['jobs']:>8}")

grid                    maps   fits   seconds    jobs
maxDepth x maxIter         4     13       7.2     417
maxDepth only              2      7       4.7     288


The job count fell with the grid, and it did not fall for caching or for parallelism. That is the
whole distinction in one table.

The fall is not exactly a half, and the reason is worth noticing: the two maps that were dropped
were the `maxIter=5` ones, which were the *cheaper* half. Halving a grid halves the number of
fits, not the amount of work, unless every point of the grid costs the same — and points of a
grid over model complexity never do.

---

## 6. The arithmetic, projected

The number of fits is the size of the grid times the number of folds, plus one. It grows
multiplicatively in the number of parameters and only additively in the size of the cluster, so
the two are not interchangeable responses to a search that is too slow.

In [11]:
print(f"{'parameters':>11}{'values each':>13}{'maps':>7}{'folds':>7}{'fits':>8}")
for params, values, folds in [(2, 2, 3), (2, 3, 3), (3, 3, 5), (4, 3, 10), (5, 3, 10)]:
    maps = values ** params
    print(f"{params:>11}{values:>13}{maps:>7}{folds:>7}{maps * folds + 1:>8}")

 parameters  values each   maps  folds    fits
          2            2      4      3      13
          2            3      9      3      28
          3            3     27      5     136
          4            3     81     10     811
          5            3    243     10    2431


In [12]:
# The chapter's two worked cases, checked.
assert (3 * 2) * 2 == 12                       # the documentation's own example
assert 4 * 3 == 12 and 4 * 3 + 1 == 13         # this chapter's grid at the default three folds
assert 3 ** 4 == 81 and 81 * 10 == 810         # four parameters, three values each, ten folds
print("(3 x 2) x 2 = 12 |  4 x 3 = 12 fits + 1 refit |  3^4 = 81 maps x 10 folds = 810 fits")

(3 x 2) x 2 = 12 |  4 x 3 = 12 fits + 1 refit |  3^4 = 81 maps x 10 folds = 810 fits


In [13]:
# Exercise 2's grid: three maxDepth, three maxIter, two stepSize, CrossValidator at its defaults.
maps = 3 * 3 * 2
print(f"maps: {maps}   folds: 3 (the default)   fits: {maps} x 3 = {maps * 3}, "
      f"plus the refit = {maps * 3 + 1}")

maps: 18   folds: 3 (the default)   fits: 18 x 3 = 54, plus the refit = 55


---

## What to take away

Measured on this machine, over one grid of four parameter maps at three folds:

* **Caching the search input** cut the wall-clock time and left the job count unchanged. Every
  fold of every map was rebuilding the DataFrame from the CSV.
* **Raising `parallelism` from 1 to 4** cut the wall-clock time further and left the job count
  unchanged again. The same work, rescheduled, competing for one memory pool.
* **Halving the grid** cut the job count, and it is the only one of the three that did. It is
  also the only one that can change which model you end up with.

Caching first, then parallelism, then the grid — because the first two are free of consequences
for the answer and the third is not.

*The seconds above are machine-dependent. The job counts are not.*

In [14]:
spark.stop()